# Feature Extraction from Preprocessed MAUS Data

Two algorithms are applied to the windowed physiological signals:

1. **Statistical + Spectral Features** — time-domain statistics and frequency-domain power band features
2. **Discrete Wavelet Transform (DWT) Features** — multi-resolution decomposition coefficients

Both operate on each window independently and produce a fixed-length feature vector per window.

In [2]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import welch
from scipy.stats import skew, kurtosis
import pywt
import warnings
warnings.filterwarnings('ignore')

# --- Load preprocessed data ---
BASE_DIR = os.path.dirname(os.path.abspath('__file__'))
PREP_DIR = os.path.join(BASE_DIR, 'preprocessed')

data = np.load(os.path.join(PREP_DIR, 'windowed_data.npz'))
splits = np.load(os.path.join(PREP_DIR, 'splits.npz'), allow_pickle=True)

X_ecg = data['X_ecg']          # (N, 1, 2560)
X_ppg_inf = data['X_ppg_inf']
X_ppg_pix = data['X_ppg_pix']
X_gsr = data['X_gsr']
y = data['y']                   # (N,)
is_clean = data['is_clean']
participants = data['participants']

FS = 256  # All signals resampled to 256 Hz

print(f"Loaded {len(y)} windows")
print(f"Clean windows: {is_clean.sum()}")
print(f"Label distribution: {dict(zip(*np.unique(y, return_counts=True)))}")

Loaded 7788 windows
Clean windows: 7788
Label distribution: {np.int64(0): np.int64(2596), np.int64(1): np.int64(2596), np.int64(2): np.int64(2596)}


---
## Algorithm 1: Statistical + Spectral Features

For each 10-second window, extract:

**Time-domain (7 features per channel):**
- Mean, Standard Deviation, RMS
- Skewness, Kurtosis
- Peak-to-peak amplitude
- Zero-crossing rate

**Frequency-domain (6 features per channel):**
- Power in signal-specific frequency bands (VLF, LF, HF)
- LF/HF ratio
- Dominant frequency
- Spectral entropy

In [ ]:
# --- Frequency band definitions per signal type ---
# These are standard physiological frequency bands
FREQ_BANDS = {
    'ecg': {
        'VLF': (0.003, 0.04),   # Very low frequency
        'LF':  (0.04, 0.15),    # Low frequency (sympathetic + parasympathetic)
        'HF':  (0.15, 0.4),     # High frequency (parasympathetic / respiratory)
    },
    'ppg': {
        'VLF': (0.003, 0.04),
        'LF':  (0.04, 0.15),
        'HF':  (0.15, 0.4),
    },
    'gsr': {
        'Tonic':  (0.05, 0.2),   # Slow tonic component
        'Phasic': (0.2, 1.0),    # Faster phasic SCR responses
        'Fast':   (1.0, 5.0),    # High-freq noise / motion artifacts
    },
}


def compute_band_power(signal, fs, band):
    """Compute power in a specific frequency band using Welch's method."""
    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), 512))
    band_mask = (freqs >= band[0]) & (freqs <= band[1])
    return np.trapezoid(psd[band_mask], freqs[band_mask]) if band_mask.any() else 0.0


def spectral_entropy(signal, fs):
    """Compute normalized spectral entropy."""
    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), 512))
    psd_norm = psd / (psd.sum() + 1e-12)
    entropy = -np.sum(psd_norm * np.log2(psd_norm + 1e-12))
    return entropy / np.log2(len(psd))  # Normalize to [0, 1]


def dominant_frequency(signal, fs):
    """Find the frequency with maximum power."""
    freqs, psd = welch(signal, fs=fs, nperseg=min(len(signal), 512))
    return freqs[np.argmax(psd)]


def zero_crossing_rate(signal):
    """Fraction of consecutive samples that cross zero."""
    return np.sum(np.diff(np.sign(signal)) != 0) / len(signal)


def extract_stat_spectral_features(signal, fs, signal_type):
    """
    Extract statistical + spectral features from a single-channel window.
    
    Returns: feature dict with named features.
    """
    features = {}
    
    # --- Time-domain ---
    features['mean'] = np.mean(signal)
    features['std'] = np.std(signal)
    features['rms'] = np.sqrt(np.mean(signal ** 2))
    features['skewness'] = skew(signal)
    features['kurtosis'] = kurtosis(signal)
    features['peak_to_peak'] = np.ptp(signal)
    features['zcr'] = zero_crossing_rate(signal)
    
    # --- Frequency-domain ---
    band_type = 'gsr' if signal_type == 'gsr' else ('ecg' if signal_type == 'ecg' else 'ppg')
    bands = FREQ_BANDS[band_type]
    band_names = list(bands.keys())
    
    band_powers = {}
    for bname, brange in bands.items():
        bp = compute_band_power(signal, fs, brange)
        features[f'power_{bname}'] = bp
        band_powers[bname] = bp
    
    # Ratio of first two bands (LF/HF or Tonic/Phasic)
    b1, b2 = band_names[1], band_names[2]
    features[f'{b1}_{b2}_ratio'] = band_powers[b1] / (band_powers[b2] + 1e-12)
    
    features['dominant_freq'] = dominant_frequency(signal, fs)
    features['spectral_entropy'] = spectral_entropy(signal, fs)
    
    return features


# Quick test on one window
test_feats = extract_stat_spectral_features(X_ecg[0, 0, :], FS, 'ecg')
print(f"Algorithm 1 produces {len(test_feats)} features per channel:")
for k, v in test_feats.items():
    print(f"  {k}: {v:.6f}")

In [4]:
# --- Extract Algorithm 1 features for all clean windows, all modalities ---

modalities = {
    'ecg':     (X_ecg, 'ecg'),
    'ppg_inf': (X_ppg_inf, 'ppg'),
    'ppg_pix': (X_ppg_pix, 'ppg'),
    'gsr':     (X_gsr, 'gsr'),
}

clean_idx = np.where(is_clean)[0]
print(f"Extracting Stat+Spectral features for {len(clean_idx)} clean windows...")

all_feat_rows = []
for i, widx in enumerate(clean_idx):
    row = {'window_idx': widx, 'label': y[widx], 'participant': participants[widx]}
    
    for mod_name, (X_mod, sig_type) in modalities.items():
        feats = extract_stat_spectral_features(X_mod[widx, 0, :], FS, sig_type)
        for fname, fval in feats.items():
            row[f'{mod_name}_{fname}'] = fval
    
    all_feat_rows.append(row)
    
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(clean_idx)} done")

df_algo1 = pd.DataFrame(all_feat_rows)

# Separate features from metadata
meta_cols = ['window_idx', 'label', 'participant']
feat_cols_algo1 = [c for c in df_algo1.columns if c not in meta_cols]

print(f"\nAlgorithm 1 complete: {df_algo1.shape[0]} windows x {len(feat_cols_algo1)} features")
print(f"Feature columns: {feat_cols_algo1[:10]} ... ({len(feat_cols_algo1)} total)")

Extracting Stat+Spectral features for 7788 clean windows...


AttributeError: module 'numpy' has no attribute 'trapz'

---
## Algorithm 2: Discrete Wavelet Transform (DWT) Features

Decompose each window using a multi-level DWT (Daubechies-4 wavelet, 5 levels).
At each decomposition level, extract:
- **Energy** of the detail/approximation coefficients
- **Entropy** (Shannon)
- **Mean** and **Std** of coefficients

This captures multi-resolution time-frequency information that spectral features miss.

In [ ]:
DWT_WAVELET = 'db4'   # Daubechies-4, good for physiological signals
DWT_LEVEL = 5          # 5 levels of decomposition


def coeff_entropy(coeffs):
    """Shannon entropy of squared wavelet coefficients."""
    c2 = coeffs ** 2
    c2_norm = c2 / (c2.sum() + 1e-12)
    return -np.sum(c2_norm * np.log2(c2_norm + 1e-12))


def extract_dwt_features(signal, wavelet=DWT_WAVELET, level=DWT_LEVEL):
    """
    Decompose signal with DWT and extract features at each level.
    
    Decomposition produces:
      [cA5, cD5, cD4, cD3, cD2, cD1]
       ^approx  ^---- detail coefficients ----^
    
    Returns: dict of named features.
    """
    coeffs = pywt.wavedec(signal, wavelet, level=level)
    # coeffs[0] = approximation at level 5 (lowest freq)
    # coeffs[1..5] = detail at levels 5,4,3,2,1 (low to high freq)
    
    features = {}
    total_energy = sum(np.sum(c ** 2) for c in coeffs) + 1e-12
    
    for i, c in enumerate(coeffs):
        if i == 0:
            name = f'A{level}'  # Approximation
        else:
            name = f'D{level - i + 1}'  # Detail level
        
        energy = np.sum(c ** 2)
        features[f'{name}_energy'] = energy
        features[f'{name}_energy_ratio'] = energy / total_energy
        features[f'{name}_entropy'] = coeff_entropy(c)
        features[f'{name}_mean'] = np.mean(c)
        features[f'{name}_std'] = np.std(c)
    
    return features


# Quick test
test_dwt = extract_dwt_features(X_ecg[0, 0, :])
print(f"Algorithm 2 produces {len(test_dwt)} features per channel:")
for k, v in test_dwt.items():
    print(f"  {k}: {v:.6f}")

print(f"\nFrequency ranges covered at 256 Hz (approximate):")
for lv in range(1, DWT_LEVEL + 1):
    lo = FS / (2 ** (lv + 1))
    hi = FS / (2 ** lv)
    print(f"  D{lv}: {lo:.1f} - {hi:.1f} Hz")
print(f"  A{DWT_LEVEL}: 0 - {FS / (2 ** (DWT_LEVEL + 1)):.1f} Hz")

In [ ]:
# --- Extract Algorithm 2 features for all clean windows, all modalities ---

print(f"Extracting DWT features for {len(clean_idx)} clean windows...")

dwt_feat_rows = []
for i, widx in enumerate(clean_idx):
    row = {'window_idx': widx, 'label': y[widx], 'participant': participants[widx]}
    
    for mod_name, (X_mod, _) in modalities.items():
        feats = extract_dwt_features(X_mod[widx, 0, :])
        for fname, fval in feats.items():
            row[f'{mod_name}_{fname}'] = fval
    
    dwt_feat_rows.append(row)
    
    if (i + 1) % 200 == 0:
        print(f"  {i+1}/{len(clean_idx)} done")

df_algo2 = pd.DataFrame(dwt_feat_rows)

feat_cols_algo2 = [c for c in df_algo2.columns if c not in meta_cols]

print(f"\nAlgorithm 2 complete: {df_algo2.shape[0]} windows x {len(feat_cols_algo2)} features")
print(f"Feature columns: {feat_cols_algo2[:10]} ... ({len(feat_cols_algo2)} total)")

---
## Results: Algorithm 1 — Statistical + Spectral Features

In [ ]:
# --- Summary statistics per class ---
label_names = {0: '0-back (Low)', 1: '2-back (Med)', 2: '3-back (High)'}

print("=" * 80)
print("ALGORITHM 1: Statistical + Spectral Features — Summary by Workload Class")
print("=" * 80)

summary = df_algo1.groupby('label')[feat_cols_algo1].agg(['mean', 'std'])
# Show a subset of important features
display_feats = [
    'ecg_std', 'ecg_kurtosis', 'ecg_power_LF', 'ecg_power_HF', 'ecg_LF_HF_ratio',
    'ecg_spectral_entropy', 'ppg_pix_std', 'ppg_pix_power_HF',
    'gsr_std', 'gsr_power_Phasic',
]

for feat in display_feats:
    if feat in feat_cols_algo1:
        print(f"\n{feat}:")
        for lbl in [0, 1, 2]:
            subset = df_algo1[df_algo1['label'] == lbl][feat]
            print(f"  {label_names[lbl]:20s}  mean={subset.mean():.6f}  std={subset.std():.6f}")

In [ ]:
# --- Box plots: key features by workload class ---
plot_feats = [
    'ecg_power_LF', 'ecg_power_HF', 'ecg_LF_HF_ratio', 'ecg_spectral_entropy',
    'ppg_pix_std', 'ppg_pix_spectral_entropy',
    'gsr_std', 'gsr_power_Phasic',
]
# Filter to features that exist
plot_feats = [f for f in plot_feats if f in feat_cols_algo1]

n_plots = len(plot_feats)
ncols = 4
nrows = (n_plots + ncols - 1) // ncols

fig, axes = plt.subplots(nrows, ncols, figsize=(16, 4 * nrows))
axes = axes.flatten()

for idx, feat in enumerate(plot_feats):
    ax = axes[idx]
    box_data = [df_algo1[df_algo1['label'] == lbl][feat].values for lbl in [0, 1, 2]]
    bp = ax.boxplot(box_data, labels=['0-back', '2-back', '3-back'], patch_artist=True)
    colors = ['#2ecc71', '#f39c12', '#e74c3c']
    for patch, color in zip(bp['boxes'], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.6)
    ax.set_title(feat, fontsize=10)
    ax.set_ylabel('Value')

# Hide unused axes
for idx in range(n_plots, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Algorithm 1: Statistical + Spectral Features by Workload', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PREP_DIR, 'algo1_boxplots.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Correlation heatmap of Algorithm 1 features (ECG subset) ---
ecg_feat_cols = [c for c in feat_cols_algo1 if c.startswith('ecg_')]

fig, ax = plt.subplots(figsize=(10, 8))
corr = df_algo1[ecg_feat_cols].corr()
im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
ax.set_xticks(range(len(ecg_feat_cols)))
ax.set_yticks(range(len(ecg_feat_cols)))
short_names = [c.replace('ecg_', '') for c in ecg_feat_cols]
ax.set_xticklabels(short_names, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(short_names, fontsize=8)
plt.colorbar(im, ax=ax, label='Pearson r')
ax.set_title('Algorithm 1: ECG Feature Correlation Matrix')
plt.tight_layout()
plt.savefig(os.path.join(PREP_DIR, 'algo1_correlation.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Results: Algorithm 2 — DWT Features

In [ ]:
print("=" * 80)
print("ALGORITHM 2: Discrete Wavelet Transform Features — Summary by Workload Class")
print("=" * 80)

display_dwt = [
    'ecg_D1_energy_ratio', 'ecg_D2_energy_ratio', 'ecg_D3_energy_ratio',
    'ecg_D4_energy_ratio', 'ecg_D5_energy_ratio', 'ecg_A5_energy_ratio',
    'ppg_pix_D3_energy_ratio', 'gsr_D5_energy_ratio',
]

for feat in display_dwt:
    if feat in feat_cols_algo2:
        print(f"\n{feat}:")
        for lbl in [0, 1, 2]:
            subset = df_algo2[df_algo2['label'] == lbl][feat]
            print(f"  {label_names[lbl]:20s}  mean={subset.mean():.6f}  std={subset.std():.6f}")

In [ ]:
# --- Stacked bar: energy distribution across DWT levels per class ---

fig, axes = plt.subplots(1, 4, figsize=(20, 5))
mod_names_display = {'ecg': 'ECG', 'ppg_inf': 'Fingertip PPG', 'ppg_pix': 'Wrist PPG', 'gsr': 'GSR'}

for ax, mod_name in zip(axes, ['ecg', 'ppg_inf', 'ppg_pix', 'gsr']):
    level_names = [f'D1', 'D2', 'D3', 'D4', 'D5', f'A5']
    energy_cols = [f'{mod_name}_{ln}_energy_ratio' for ln in level_names]
    energy_cols = [c for c in energy_cols if c in feat_cols_algo2]
    
    x_pos = np.arange(3)
    bottoms = np.zeros(3)
    colors_levels = plt.cm.viridis(np.linspace(0.2, 0.9, len(energy_cols)))
    
    for col, color in zip(energy_cols, colors_levels):
        means = [df_algo2[df_algo2['label'] == lbl][col].mean() for lbl in [0, 1, 2]]
        short = col.replace(f'{mod_name}_', '').replace('_energy_ratio', '')
        ax.bar(x_pos, means, bottom=bottoms, label=short, color=color, width=0.6)
        bottoms += means
    
    ax.set_xticks(x_pos)
    ax.set_xticklabels(['0-back', '2-back', '3-back'])
    ax.set_ylabel('Energy Ratio')
    ax.set_title(mod_names_display[mod_name])
    ax.legend(fontsize=8, loc='upper right')
    ax.set_ylim(0, 1.1)

plt.suptitle('Algorithm 2: DWT Energy Distribution by Decomposition Level and Workload', fontsize=13)
plt.tight_layout()
plt.savefig(os.path.join(PREP_DIR, 'algo2_energy_bars.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- DWT entropy across levels for ECG by class ---

fig, ax = plt.subplots(figsize=(10, 5))

level_labels = ['D1\n(64-128Hz)', 'D2\n(32-64Hz)', 'D3\n(16-32Hz)',
                'D4\n(8-16Hz)', 'D5\n(4-8Hz)', 'A5\n(0-4Hz)']
entropy_cols = [f'ecg_{ln}_entropy' for ln in ['D1', 'D2', 'D3', 'D4', 'D5', 'A5']]
entropy_cols = [c for c in entropy_cols if c in feat_cols_algo2]

colors_class = ['#2ecc71', '#f39c12', '#e74c3c']
x = np.arange(len(entropy_cols))
width = 0.25

for lbl, color in zip([0, 1, 2], colors_class):
    means = [df_algo2[df_algo2['label'] == lbl][col].mean() for col in entropy_cols]
    stds = [df_algo2[df_algo2['label'] == lbl][col].std() for col in entropy_cols]
    ax.bar(x + lbl * width, means, width, yerr=stds, label=label_names[lbl],
           color=color, alpha=0.7, capsize=3)

ax.set_xticks(x + width)
ax.set_xticklabels(level_labels[:len(entropy_cols)])
ax.set_ylabel('Shannon Entropy')
ax.set_title('Algorithm 2: ECG Wavelet Coefficient Entropy by Level and Workload')
ax.legend()
plt.tight_layout()
plt.savefig(os.path.join(PREP_DIR, 'algo2_entropy.png'), dpi=150, bbox_inches='tight')
plt.show()

---
## Comparison: Both Algorithms Side by Side

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
colors_map = {0: '#2ecc71', 1: '#f39c12', 2: '#e74c3c'}

# --- PCA on Algorithm 1 features ---
X_a1 = df_algo1[feat_cols_algo1].values
X_a1 = np.nan_to_num(X_a1, nan=0.0, posinf=0.0, neginf=0.0)
X_a1_scaled = StandardScaler().fit_transform(X_a1)
pca1 = PCA(n_components=2)
Z1 = pca1.fit_transform(X_a1_scaled)

ax = axes[0]
for lbl in [0, 1, 2]:
    mask = df_algo1['label'].values == lbl
    ax.scatter(Z1[mask, 0], Z1[mask, 1], c=colors_map[lbl],
               label=label_names[lbl], alpha=0.4, s=10)
ax.set_xlabel(f'PC1 ({pca1.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca1.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('Algorithm 1: Stat+Spectral — PCA')
ax.legend(markerscale=3)

# --- PCA on Algorithm 2 features ---
X_a2 = df_algo2[feat_cols_algo2].values
X_a2 = np.nan_to_num(X_a2, nan=0.0, posinf=0.0, neginf=0.0)
X_a2_scaled = StandardScaler().fit_transform(X_a2)
pca2 = PCA(n_components=2)
Z2 = pca2.fit_transform(X_a2_scaled)

ax = axes[1]
for lbl in [0, 1, 2]:
    mask = df_algo2['label'].values == lbl
    ax.scatter(Z2[mask, 0], Z2[mask, 1], c=colors_map[lbl],
               label=label_names[lbl], alpha=0.4, s=10)
ax.set_xlabel(f'PC1 ({pca2.explained_variance_ratio_[0]:.1%} var)')
ax.set_ylabel(f'PC2 ({pca2.explained_variance_ratio_[1]:.1%} var)')
ax.set_title('Algorithm 2: DWT — PCA')
ax.legend(markerscale=3)

plt.suptitle('Feature Space Comparison: PCA Projection by Workload Class', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(PREP_DIR, 'feature_pca_comparison.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- Feature count and dimensionality summary ---

print("=" * 70)
print("FEATURE EXTRACTION SUMMARY")
print("=" * 70)
print(f"")
print(f"Algorithm 1: Statistical + Spectral Features")
print(f"  Features per channel:  13")
print(f"  Channels:              4 (ECG, PPG_inf, PPG_pix, GSR)")
print(f"  Total features:        {len(feat_cols_algo1)}")
print(f"  PCA variance (2 PCs):  {pca1.explained_variance_ratio_.sum():.1%}")
print(f"")
print(f"Algorithm 2: Discrete Wavelet Transform")
print(f"  DWT levels:            {DWT_LEVEL} ({DWT_WAVELET} wavelet)")
print(f"  Features per channel:  {DWT_LEVEL + 1} levels x 5 stats = {(DWT_LEVEL + 1) * 5}")
print(f"  Channels:              4")
print(f"  Total features:        {len(feat_cols_algo2)}")
print(f"  PCA variance (2 PCs):  {pca2.explained_variance_ratio_.sum():.1%}")
print(f"")
print(f"Windows processed:       {len(clean_idx)} (clean only)")
print(f"Classes:                 3 (0-back / 2-back / 3-back)")

In [ ]:
# --- Save extracted feature tables ---

df_algo1.to_csv(os.path.join(PREP_DIR, 'features_stat_spectral.csv'), index=False)
df_algo2.to_csv(os.path.join(PREP_DIR, 'features_dwt.csv'), index=False)

# Also save as npz for model training
np.savez_compressed(
    os.path.join(PREP_DIR, 'features_extracted.npz'),
    algo1_features=X_a1.astype(np.float32),
    algo2_features=X_a2.astype(np.float32),
    labels=df_algo1['label'].values,
    participants=df_algo1['participant'].values,
    algo1_feature_names=np.array(feat_cols_algo1),
    algo2_feature_names=np.array(feat_cols_algo2),
)

print(f"Saved to {PREP_DIR}/")
print(f"  features_stat_spectral.csv  ({df_algo1.shape})")
print(f"  features_dwt.csv            ({df_algo2.shape})")
print(f"  features_extracted.npz")